In [0]:
from datetime import datetime
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,DecimalType,LongType,DateType
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import *
import sys

In [0]:
dbutils.widgets.text(name='Environemnt',defaultValue='devv')

dbutils.widgets.get('Environemnt')

In [0]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

In [0]:
# calling the notebook to create the mount points 
notebook_name = "Sales_Data_processing"
try:
    dbutils.notebook.run(path='/Workspace/Shared/Sales_Data/python_notebooks/Mount_points',timeout_seconds=120)
    dbutils.notebook.run(path='/Workspace/Shared/Sales_Data/sqlfiles/exception_table_creation',timeout_seconds=120)
    dbutils.notebook.run(path='/Workspace/Shared/Sales_Data/sqlfiles/Sales_Import_table_creation',timeout_seconds=120)
    dbutils.notebook.run(path='/Workspace/Shared/Sales_Data/sqlfiles/logs_table_creation',timeout_seconds=120)
    dbutils.notebook.run(path='/Workspace/Shared/Sales_Data/sqlfiles/tbl_sales_I2004_output',timeout_seconds=120)

except Exception as E:
    spark.sql(f"""
              insert into default.dgrs_notebook_logs(notebook_name,exceuted_by,status,cluster_name )
               values ('{notebook_name}','srinivas.mentula','failed','default')""")
    raise E



In [0]:
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", None)

print(cluster_id)

In [0]:
# Creating the file path 
try:
    current_day = datetime.now()
    year = current_day.year
    month = current_day.month
    day = current_day.day
    current_day_name = datetime.today().strftime('%A')
    file_path_locatiion = f"/mnt/inbound/{year}/{month:02d}/{day:02d}"
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", None)
    # creating the user defined schema 

    inbound_schema = StructType([
                                    StructField("storeId",IntegerType(),True),
                                    StructField("busday",DateType(),True),
                                    StructField("WMkeyN1",IntegerType(),True),
                                    StructField("saparticle",IntegerType(),True),
                                    StructField("promo",StringType(),True),
                                    StructField("channel",StringType(),True),
                                    StructField("type",IntegerType(),True),
                                    StructField("normamt",DecimalType(9,2),True),
                                    StructField("saleamt",DecimalType(9,2),True),
                                    StructField("qty",IntegerType(),True),
                                    StructField("uom",StringType(),True),
                                    StructField("currency",StringType(),True),
                                    StructField("groupingcode",StringType(),True),
                                    StructField("digitgln_13",LongType(),True),
                                    StructField("supplier",IntegerType(),True),
                                    StructField("purchasegroup",IntegerType(),True),
                                    StructField("item1_desc",StringType(),True),
                                    StructField("item2_desc",StringType(),True),
                                    StructField("shlflbl1_colr_desc",StringType(),True),
                                    StructField("barcode",LongType(),True)
                                    ])
    # Reading the Inbound data 
    inbound_df = (spark.read.format("csv").
                        option("inferSchema",True).
                        schema(inbound_schema).
                        option("header",True).load(f"{file_path_locatiion}")

    )

  
except Exception as E:
    print(E)

In [0]:
def test_data_creation(df,multiplication_factor,key_column):
    df = df.withColumn("array_repeat",array_repeat(f"{key_column}",multiplication_factor))
    df = df.withColumn("array_repeat",explode(col("array_repeat")))
    # display(employees_data_frame_2)
    window_spec = Window.partitionBy().orderBy(col(f"{key_column}").desc())
    df = df.withColumn("row_number",row_number().over(window_spec)).drop('array_repeat')
    df = df .withColumn(key_column,col(f"{key_column}")+col("row_number"))
    return df

In [0]:
inbound_df = test_data_creation(df=inbound_df,multiplication_factor= 6000,key_column= "WMkeyN1")

In [0]:
inbound_df.write.format("csv").save("dbfs:/FileStore/employees.csv")

In [0]:
inbound_df.rdd.getNumPartitions()

In [0]:
inbound_df = inbound_df.repartition(4)

In [0]:
inbound_df.write.format("csv").save("dbfs:/FileStore/employees.csv")

In [0]:
df_size_bytes = inbound_df.select(
    sum(length(struct([col(c) for c in inbound_df.columns]).cast("binary")))
).collect()[0][0]

df_size_mb = df_size_bytes / (1024 * 1024)
print(f"Estimated DataFrame size: {df_size_mb:.2f} MB")

In [0]:
(inbound_df.write.format("delta").mode("overwrite")\
                                    .saveAsTable("default.sales_import")
    )

delta_instance  = DeltaTable.forName(spark,"default.sales_import")

I0063_table_data_frame_import  = delta_instance.toDF()

config_table_instance = DeltaTable.forName(spark,"default.grs_configuration_table")

config_data_frame  = config_table_instance.toDF()

I0063_table_data_frame_import = I0063_table_data_frame_import.alias("t1") \
    .join(
        broadcast(config_data_frame).alias("t2"),
        on=array_contains(
            split(col("t2.valid_stores"), ","), 
            col("t1.storeId").cast("string")
        )
    )

In [0]:
delta_instance  = DeltaTable.forName(spark,"default.sales_import")

I0063_table_data_frame_import  = delta_instance.toDF()


window_spec = Window.partitionBy(col("storeId"),col("wmKeyN1"))

I0063_table_data_frame_import = (I0063_table_data_frame_import.withColumn("busday",max("busday")
                                    .over(window_spec)).select(col("busday"),"storeId","WMkeyN1","qty","saleamt"))

I0063_table_data_frame_import = (
                        I0063_table_data_frame_import.groupBy(col("storeId"),col("WMkeyN1"),col("busday"))
                        .agg(sum(col("qty")).alias("total_qty"),sum(col("saleamt")).alias("sales_amt"))
)


In [0]:
delta_instance  = DeltaTable.forName(spark,"default.sales_import")

I0063_table_data_frame_import  = delta_instance.toDF()


window_spec = Window.partitionBy(col("storeId"),col("wmKeyN1"))

I0063_table_data_frame_import = (I0063_table_data_frame_import.withColumn("busday",max("busday")
                                    .over(window_spec)).select(col("busday"),"storeId","WMkeyN1","qty","saleamt"))

I0063_table_data_frame_import = (
                        I0063_table_data_frame_import.groupBy(col("storeId"),col("WMkeyN1"),col("busday"))
                        .agg(sum(col("qty")).alias("total_qty"),sum(col("saleamt")).alias("sales_amt"))
)
I0063_table_data_frame_import= (I0063_table_data_frame_import
            .withColumnRenamed("WMkeyN1","item_nbr")
            .withColumnRenamed("storeId","store_nbr")
            .withColumn("wm_yr_week",lit(1234))
            .withColumn("company_code",lit("WM"))
            .withColumn("country_code",lit('GB'))
            .withColumn("report_code",lit(0))
            .withColumn("sun_sales_amt",when(date_format(col("busday"),'EEEE')=='Sunday',col("sales_amt")).otherwise(0))
            .withColumn("mon_sales_amt",when(date_format(col("busday"),'EEEE')=='Monday',col("sales_amt")).otherwise(0))
            .withColumn("tue_sales_amt",when(date_format(col("busday"),'EEEE')=='Tuesday',col("sales_amt")).otherwise(0))
            .withColumn("wed_sales_amt",when(date_format(col("busday"),'EEEE')=='Wednesday',col("sales_amt")).otherwise(0))
            .withColumn("thu_sales_amt",when(date_format(col("busday"),'EEEE')=='Thursday',col("sales_amt")).otherwise(0))
            .withColumn("fri_sales_amt",when(date_format(col("busday"),'EEEE')=='Friday',col("sales_amt")).otherwise(0))
            .withColumn("sat_sales_amt",when(date_format(col("busday"),'EEEE')=='Saturday',col("sales_amt")).otherwise(0))
            .withColumn("mon_qty",when(date_format(col("busday"),'EEEE')=='Monday',col("total_qty")).otherwise(0))
            .withColumn("tue_qty",when(date_format(col("busday"),'EEEE')=='Tuesday',col("total_qty")).otherwise(0))
            .withColumn("wed_qty",when(date_format(col("busday"),'EEEE')=='Wednesday',col("total_qty")).otherwise(0))
            .withColumn("thu_qty",when(date_format(col("busday"),'EEEE')=='Thursday',col("total_qty")).otherwise(0))
            .withColumn("fri_qty",when(date_format(col("busday"),'EEEE')=='Friday',col("total_qty")).otherwise(0))
            .withColumn("sat_qty",when(date_format(col("busday"),'EEEE')=='Saturday',col("total_qty")).otherwise(0))
            .withColumn("sun_qty",when(date_format(col("busday"),'EEEE')=='Sunday',col("total_qty")).otherwise(0))
            )

In [0]:
I0063_table_data_frame_import = I0063_table_data_frame_import.repartition(4)


In [0]:
I0063_table_data_frame_import.rdd.getNumPartitions()

In [0]:
final_table_instance = DeltaTable.forName(spark, "default.tbl_sales_i2004_output")
columns_list = final_table_instance.toDF().columns
final_column_list = {
    f"target.{i}": col(f"source.{i}")
    for i in columns_list
    if not (
        i == "id"
        or i == "create_ts"
        or i == "sell_price"
        or i == "wkly_sales"
        or i == "wkly_qty"
        or i == "update_ts"
        or i == "create_ts"
    )
}
final_column_list_update = final_column_list.copy()
final_column_list_update["target.update_ts"] = current_timestamp()
final_column_list_insert = final_column_list.copy()
final_column_list_insert["target.create_ts"] = current_timestamp()
if current_day_name == 'Saturday':
    spark.sql("Truncate table default.tbl_sales_i2004_output ")
else:
    final_table_instance.alias("target").merge(
        source=I0063_table_data_frame_import.alias("source"),
        condition=(col("target.store_nbr") == col("source.store_nbr"))
        & (col("target.item_nbr") == col("source.item_nbr")),
    ).whenMatchedUpdate(set=final_column_list_update).whenNotMatchedInsert(
        values=final_column_list_insert
    ).execute()
    final_table_instance = DeltaTable.forName(spark, "default.tbl_sales_i2004_output")
    final_table_instance.update(
        set={
            "wkly_sales": col("mon_sales_amt")
            + col("sun_sales_amt")
            + col("tue_sales_amt")
            + col("wed_sales_amt")
            + col("thu_sales_amt")
            + col("fri_sales_amt")
            + col("sat_sales_amt"),
            "wkly_qty": col("sun_qty")
            + col("mon_qty")
            + col("tue_qty")
            + col("wed_qty")
            + col("thu_qty")
            + col("fri_qty")
            + col("sat_qty"),
        }
    )
    spark.sql(
            "update default.tbl_sales_i2004_output set sell_price = wkly_sales/if(wkly_qty=0,null,wkly_qty)"
        )

In [0]:
def estimate_partition_size(iterator):
    rows = list(iterator)return [sum(sys.getsizeof(row) for row in rows)]


df_size_bytes = sum(inbound_df.rdd.mapPartitions(estimate_partition_size).collect())
df_size_mb = df_size_bytes / (1024 * 1024)

In [0]:
result = inbound_df.rdd.mapPartitions(lambda partition: [sum(1 for i in partition)]).collect()


In [0]:
inbound_df.count()

In [0]:
def count_partition_rows(partition):
    try:
        yield sum(1 for _ in partition)
    except Exception as e:
        print("Error in partition:", e)
        yield 0

result = inbound_df.rdd.mapPartitions(count_partition_rows).collect()

print(result)


In [0]:
def count_partition_rows(partition):
    count = 0
    for _ in partition:
        count += 1
    yield count

result = inbound_df.rdd.mapPartitions(count_partition_rows).collect()
print(result)


In [0]:
inbound_df.rdd.getNumPartitions()
inbound_df = inbound_df.repartition(6)

In [0]:
# Get row counts per partition (Spark 3.0+)
inbound_df.withColumn("partition_id", spark_partition_id()) \
          .groupBy("partition_id").count() \
          .show()

In [0]:
# Just see partition IDs (no row counts)
inbound_df.rdd.glom().map(len).collect()